<a href="https://colab.research.google.com/github/dennisddschulz/cas-artificial-intelligence/blob/main/08_drl_einstieg/01_DRL_Einstieg_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DRL Intro: MDP, Return, V/Q/Advantage – Monte-Carlo Evaluation (FrozenLake)

Ziel:
1) Episode sammeln (Rollout)
2) Returns G_t berechnen
3) Monte-Carlo Schätzung von V(s) und Q(s,a)
4) Advantage A(s,a) berechnen
5) Aus Q eine ε-greedy Policy ableiten
6) Zeigen, dass sich V(start) verbessert (Policy Improvement)


In [2]:
!pip -q install gymnasium

import gymnasium as gym
import numpy as np
from collections import defaultdict

SEED = 42
rng = np.random.default_rng(SEED)


In [3]:
# Environment: deterministisch für klare Ergebnisse
env = gym.make("FrozenLake-v1", is_slippery=False)

s0, info = env.reset(seed=SEED)
nS = env.observation_space.n
nA = env.action_space.n

print("nS:", nS, "nA:", nA, "start:", s0)


nS: 16 nA: 4 start: 0


## Policy und Rollout

- Policy: Funktion, die aus Zustand s eine Aktion a wählt.
- Rollout: wir lassen Agent+Env laufen und speichern (s, a, r) pro Schritt.


In [4]:
def random_policy(s, nA):
    return int(rng.integers(nA))

def rollout_episode(env, policy_fn, max_steps=200, seed=0):
    traj = []  # list of (s, a, r)
    s, _ = env.reset(seed=seed)
    for _ in range(max_steps):
        a = policy_fn(s, env.action_space.n)
        s2, r, terminated, truncated, _ = env.step(a)
        traj.append((s, a, r))
        s = s2
        if terminated or truncated:
            break
    return traj

traj = rollout_episode(env, random_policy, seed=SEED)
print("episode length:", len(traj))
print("first steps:", traj[:8])


episode length: 4
first steps: [(0, 0, 0), (0, 3, 0), (0, 2, 0), (1, 1, 0)]


## Returns berechnen

Return G_t ist die discounted Summe der zukünftigen Rewards ab Schritt t.
Wir berechnen das rückwärts:
G = 0
G <- r + gamma * G


In [5]:
def compute_returns(traj, gamma=0.99):
    G = 0.0
    returns = []
    # HA - Warum reversed?
    for (s, a, r) in reversed(traj):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns

gamma = 0.99
Gs = compute_returns(traj, gamma=gamma)
list(zip(traj[:8], Gs[:8]))


[((0, 0, 0), 0.0), ((0, 3, 0), 0.0), ((0, 2, 0), 0.0), ((1, 1, 0), 0.0)]

## MC Evaluation von V(s)

First-Visit MC:
- pro Episode zählt nur das erste Auftreten eines Zustands s
- V(s) = Durchschnitt der beobachteten Returns in s

```
seen = set()
for t, (s, a, r) in enumerate(traj):
    if s in seen:
        continue
    seen.add(s)
    V[s] += G[t]
```

Every-Visit MC:
- pro Episode zählt jedes Auftreten eines Zustands s
- V(s) ist der Durchschnitt der beobachteten Return über alle Besuche von s in allen Episoden.
```
for t, (s, a, r) in enumerate(traj):
    V[s] += G[t]
```


In [20]:
def mc_evaluate_V_every_visit(env, policy_fn, episodes=3000, gamma=0.99, seed=0):
  returns_sum = defaultdict(float)
  returns_count = defaultdict(int)
  for ep in range(episodes):
    traj = rollout_episode(env, policy_fn, seed=seed + ep)
    Gs = compute_returns(traj, gamma=gamma)
    for t, (s, a, r) in enumerate(traj):
      returns_sum[s] += Gs[t]
      returns_count[s] += 1
      return {s: returns_sum[s] / returns_count[s] for s in returns_count}
V_rand = mc_evaluate_V_every_visit(env, random_policy, episodes=4000, gamma=gamma, seed=SEED)

s0, _ = env.reset(seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))

V_random(start): 0.0


In [23]:
def mc_evaluate_V(env, policy_fn, episodes=3000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        seen = set()
        for t, (s, a, r) in enumerate(traj):
            if s in seen:
                continue
            seen.add(s)
            returns_sum[s] += Gs[t]
            returns_count[s] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_count}
    return V

V_rand = mc_evaluate_V(env, random_policy, episodes=4000, gamma=gamma, seed=SEED)

s0, _ = env.reset(seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))


V_random(start): 0.0101


In [7]:
V_rand

{0: 0.01243207145196423,
 4: 0.01748787153372368,
 1: 0.007234537243722057,
 2: 0.012189950638019269,
 3: 0.0020033272971132473,
 6: 0.026821612870081044,
 8: 0.03845824015901537,
 9: 0.08792815679695472,
 10: 0.12013856417409896,
 13: 0.18086616947046652,
 14: 0.40231443517025645}

## MC Evaluation von Q(s,a)

Analog:
- wir mitteln Returns pro (s,a)
- daraus können wir greedy / ε-greedy Policies bauen


In [8]:
def mc_evaluate_Q(env, policy_fn, episodes=6000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        seen_sa = set()
        for t, (s, a, r) in enumerate(traj):
            key = (s, a)
            if key in seen_sa:
                continue
            seen_sa.add(key)
            returns_sum[key] += Gs[t]
            returns_count[key] += 1

    Q = {k: returns_sum[k] / returns_count[k] for k in returns_count}
    return Q

Q_rand = mc_evaluate_Q(env, random_policy, episodes=8000, gamma=gamma, seed=SEED)
print("Q entries:", list(Q_rand.items())[:5])


Q entries: [((0, 1), 0.014838457167547595), ((4, 0), 0.017789925010889333), ((4, 3), 0.01565071370145109), ((4, 2), 0.0), ((0, 2), 0.010242417311593928)]


## Advantage A(s,a)

A(s,a) = Q(s,a) - V(s)
Interpretation: wie viel besser/schlechter ist Aktion a gegenüber dem "Durchschnitt" in s.


In [9]:
def advantage(V, Q, s, a):
    return Q.get((s, a), 0.0) - V.get(s, 0.0)

# Advantage im Startzustand für alle Aktionen
adv_start = [(a, advantage(V_rand, Q_rand, s0, a)) for a in range(nA)]
adv_start


[(0, 3.3547907898329524e-05),
 (1, 0.0024063857155833656),
 (2, -0.0021896541403703014),
 (3, -0.0002846759044804973)]

## Policy Improvement: ε-greedy aus Q

- greedy: a = argmax_a Q(s,a)
- ε-greedy: mit Wahrscheinlichkeit ε zufällig, sonst greedy

Dann evaluieren wir die neue Policy wieder mit MC und vergleichen V(start).


In [10]:
def epsilon_greedy_policy_from_Q(Q, nA, eps=0.1):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        qs = [Q.get((s, a), 0.0) for a in range(nA)]
        return int(np.argmax(qs))
    return policy

pi_eps = epsilon_greedy_policy_from_Q(Q_rand, nA, eps=0.1)

V_eps = mc_evaluate_V(env, pi_eps, episodes=4000, gamma=gamma, seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))
print("V_eps(start):   ", round(V_eps.get(s0, 0.0), 4))


V_random(start): 0.0124
V_eps(start):    0.8472


## Mini Loop: wiederholte Verbesserung (Iteration)

Wir wiederholen:
1) Q unter aktueller Policy schätzen
2) neue ε-greedy Policy bauen
3) V(start) loggen

Achtung: Das ist noch nicht "Policy Iteration" im strengen Sinn,
aber zeigt sehr gut die Grundidee: Bessere Wertschätzungen (V/Q) → bessere Entscheidungsgrundlage → verbesserte Policy


In [11]:
def policy_improvement_loop(env, init_policy, iters=5, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=0.99, seed=0):
    policy = init_policy
    history = []

    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)
        policy = epsilon_greedy_policy_from_Q(Q, env.action_space.n, eps=eps)
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k)
        history.append((k, V.get(s0, 0.0)))

    return history

hist = policy_improvement_loop(env, random_policy, iters=6, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=gamma, seed=SEED)
hist


[(0, 0.8507260722145118),
 (1, 0.2563801026839335),
 (2, 0.8439864371191398),
 (3, 0.3926854148080327),
 (4, 0.8518339146322825),
 (5, 0.4175155124469167)]

## Was haben wir heute gelernt?

- Reward vs Return: Return ist das Ziel, nicht der einzelne Reward.
- V(s) und Q(s,a) sind Erwartungswerte von Returns.
- Monte-Carlo schätzt diese Werte aus Episoden (ohne Modell von P).
- Advantage erklärt "wie gut ist diese Aktion relativ zum Durchschnitt in s".
- Aus Q kann man eine bessere Policy ableiten (ε-greedy).
